<a href="https://colab.research.google.com/github/alchemist-sourav/LogiChain-Supply-Delay-Prediction/blob/main/LogiChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import zipfile
import os
from google.colab import files

# Upload ZIP files
uploaded = files.upload()

extract_dir = "extracted_data"
os.makedirs(extract_dir, exist_ok=True)

# Extract all ZIPs
for file_name in uploaded.keys():
    if file_name.endswith(".zip"):
        print(f"Extracting {file_name}...")
        with zipfile.ZipFile(file_name, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)

# Find all CSV files
csv_files = []

for root, dirs, files_list in os.walk(extract_dir):
    for file in files_list:
        if file.endswith(".csv"):
            csv_files.append(os.path.join(root, file))

print("\nCSV Files Found:")
for csv in csv_files:
    print(csv)

# Load and inspect
dfs = []

for csv in csv_files:
    try:
        df = pd.read_csv(csv, encoding="latin1")
        print("\n" + "="*60)
        print(f"File: {os.path.basename(csv)}")
        print(f"Shape: {df.shape}")
        print(f"Columns: {len(df.columns)}")
        print(df.columns.tolist())
        dfs.append(df)
    except Exception as e:
        print(f"Error reading {csv}: {e}")

# Combine only datasets with identical columns
combined_df = None

if len(dfs) > 0:
    base_cols = set(dfs[0].columns)

    compatible = []

    for df in dfs:
        if set(df.columns) == base_cols:
            compatible.append(df)

    if compatible:
        combined_df = pd.concat(compatible, ignore_index=True)

        print("\nCombined Dataset Shape:")
        print(combined_df.shape)

        combined_df.to_csv(
            "combined_supply_chain_dataset.csv",
            index=False
        )

        print(
            "\nSaved: combined_supply_chain_dataset.csv"
        )

Saving archive(2).zip to archive(2).zip
Saving archive(3).zip to archive(3).zip
Saving archive(4).zip to archive(4).zip
Saving archive(5).zip to archive(5).zip
Extracting archive(2).zip...
Extracting archive(3).zip...
Extracting archive(4).zip...
Extracting archive(5).zip...

CSV Files Found:
extracted_data/DescriptionDataCoSupplyChainRefined.csv
extracted_data/DataCoSupplyChainDataset.csv
extracted_data/tokenized_access_logs.csv
extracted_data/DescriptionDataCoSupplyChain.csv
extracted_data/DataCoSupplyChainDatasetRefined.csv

File: DescriptionDataCoSupplyChainRefined.csv
Shape: (58, 2)
Columns: 2
['FIELDS', 'DESCRIPTION']

File: DataCoSupplyChainDataset.csv
Shape: (180519, 53)
Columns: 53
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Custome

In [4]:
print(combined_df.shape)
print(combined_df.columns.tolist())

(110, 2)
['FIELDS', 'DESCRIPTION']


In [5]:
import pandas as pd

df = pd.read_csv(
    "extracted_data/DataCoSupplyChainDatasetRefined.csv"
)

print(df.shape)
df.head()

(180519, 58)


,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,...,product_price,product_status,shipping_date_dateorders,shipping_mode,order_country_en,order_state_en,order_city_en,latitude_dest,longitude_dest,address_dest
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,327.75,0,2/3/2018 22:56,Standard Class,Indonesia,West Java,Bekasi,-6.238270,106.975573,"Bekasi, West Java, Indonesia"
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,327.75,0,1/18/2018 12:27,Standard Class,India,Rajasthan,Bikaner,28.022935,73.311916,"Bikaner, Rajasthan, India"
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,327.75,0,1/17/2018 12:06,Standard Class,India,Rajasthan,Bikaner,28.022935,73.311916,"Bikaner, Rajasthan, India"
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,327.75,0,1/16/2018 11:45,Standard Class,Australia,Queensland,Townsville,-19.258964,146.816948,"Townsville City QLD 4810, Australia"
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,327.75,0,1/15/2018 11:24,Standard Class,Australia,Queensland,Townsville,-19.258964,146.816948,"Townsville City QLD 4810, Australia"


In [6]:
print(df["late_delivery_risk"].value_counts())

late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64


In [7]:
print(
    df["late_delivery_risk"]
    .value_counts(normalize=True) * 100
)

late_delivery_risk
1    54.829132
0    45.170868
Name: proportion, dtype: float64


In [8]:
features = [
    "shipping_mode",
    "market",
    "order_region",
    "category_name",
    "sales",
    "benefit_per_order",
    "days_for_shipment_scheduled",
    "product_price",
    "order_item_quantity"
]

target = "late_delivery_risk"

In [9]:
from sklearn.preprocessing import LabelEncoder

categorical = [
    "shipping_mode",
    "market",
    "order_region",
    "category_name"
]

encoders = {}

for col in categorical:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print(
    "Accuracy:",
    accuracy_score(y_test, pred)
)

Accuracy: 0.6198759140261467


In [11]:
import joblib

joblib.dump(
    model,
    "shipment_delay_model.pkl"
)

joblib.dump(
    encoders,
    "encoders.pkl"
)

['encoders.pkl']

In [12]:
print(df["late_delivery_risk"].value_counts())

late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64


In [13]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.58      0.58      0.58     16308
           1       0.65      0.65      0.65     19796

    accuracy                           0.62     36104
   macro avg       0.62      0.62      0.62     36104
weighted avg       0.62      0.62      0.62     36104



In [14]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, pred))

[[ 9466  6842]
 [ 6882 12914]]


In [15]:
features = [
    "shipping_mode",
    "market",
    "order_region",
    "customer_segment",
    "department_name",
    "sales",
    "benefit_per_order",
    "order_item_quantity",
    "product_price",
    "days_for_shipment_scheduled"
]

In [16]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

In [17]:
print(df["late_delivery_risk"].value_counts())
print(classification_report(y_test, pred))

late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64
              precision    recall  f1-score   support

           0       0.58      0.58      0.58     16308
           1       0.65      0.65      0.65     19796

    accuracy                           0.62     36104
   macro avg       0.62      0.62      0.62     36104
weighted avg       0.62      0.62      0.62     36104



In [18]:
df.columns.tolist()

['type',
 'days_for_shipping_real',
 'days_for_shipment_scheduled',
 'benefit_per_order',
 'sales_per_customer',
 'delivery_status',
 'late_delivery_risk',
 'category_id',
 'category_name',
 'customer_city',
 'customer_country',
 'customer_email',
 'customer_fname',
 'customer_id',
 'customer_lname',
 'customer_password',
 'customer_segment',
 'customer_state',
 'customer_street',
 'customer_zipcode',
 'department_id',
 'department_name',
 'latitude_src',
 'longitude_src',
 'market',
 'order_city',
 'order_country',
 'order_customer_id',
 'order_date_dateorders',
 'order_id',
 'order_item_cardprod_id',
 'order_item_discount',
 'order_item_discount_rate',
 'order_item_id',
 'order_item_product_price',
 'order_item_profit_ratio',
 'order_item_quantity',
 'sales',
 'order_item_total',
 'order_profit_per_order',
 'order_region',
 'order_state',
 'order_status',
 'order_zipcode',
 'product_card_id',
 'product_category_id',
 'product_image',
 'product_name',
 'product_price',
 'product_statu

In [19]:
features = [
    "shipping_mode",
    "market",
    "customer_segment",
    "department_name",
    "category_name",
    "order_region",
    "order_country",
    "sales",
    "benefit_per_order",
    "order_item_quantity",
    "product_price",
    "days_for_shipment_scheduled",
    "latitude_src",
    "longitude_src",
    "latitude_dest",
    "longitude_dest"
]

In [20]:
target = "late_delivery_risk"

In [21]:
import numpy as np

df["distance"] = np.sqrt(
    (df["latitude_dest"] - df["latitude_src"])**2 +
    (df["longitude_dest"] - df["longitude_src"])**2
)

In [22]:
"distance"

'distance'

In [23]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = [
    "shipping_mode",
    "market",
    "customer_segment",
    "department_name",
    "category_name",
    "order_region",
    "order_country"
]

encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

In [24]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

In [25]:
import joblib

joblib.dump(model, "shipment_delay_model.pkl")
joblib.dump(encoders, "encoders.pkl")

['encoders.pkl']

In [26]:
import os

print(os.listdir())


['.config', 'encoders.pkl', 'archive(5).zip', 'archive(4).zip', 'combined_supply_chain_dataset.csv', 'archive(3).zip', 'archive(2).zip', 'shipment_delay_model.pkl', 'extracted_data', 'sample_data']


In [27]:
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=8, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)

In [28]:
import joblib

joblib.dump(model, "shipment_delay_model.pkl")
joblib.dump(encoders, "encoders.pkl")

print("Files Saved")

Files Saved


In [29]:
import os

for f in os.listdir():
    if f.endswith(".pkl"):
        print(f)

encoders.pkl
shipment_delay_model.pkl


In [30]:
joblib.dump(
    {
        "model": model,
        "encoders": encoders,
        "features": features
    },
    "logischain_model_bundle.pkl"
)

['logischain_model_bundle.pkl']

In [31]:
type(model)

xgboost.sklearn.XGBClassifier

In [32]:
print(model)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=8, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)


In [33]:
import joblib

joblib.dump(model, "shipment_delay_model.pkl")

print("Model saved successfully")

Model saved successfully


In [34]:
import os

print([f for f in os.listdir() if f.endswith(".pkl")])

['encoders.pkl', 'logischain_model_bundle.pkl', 'shipment_delay_model.pkl']


In [36]:
joblib.dump(
    {
        "model": model,
        "encoders": encoders,
        "features": features
    },
    "logischain_bundle.pkl"
)

files.download("logischain_bundle.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>